A Quantitative Study on How Tariff Differences Influence Trade Reallocation

Part 1 - Data Exploration and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
import streamlit as st
import os

In [ ]:
pip install streamlit

  Using cached altair-5.5.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached GitPython-3.1.44-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 184.7 kB/s eta 0:00:0000:0100:03
Using cached altair-5.5.0-py3-none-any.whl (731 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
Using cached GitPython-3.1.44-py3-none-any.whl (207 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 28.3/30.7 MB 151.6 kB/s eta 0:00:16

In [20]:
np.random.seed(42)

In [5]:
tarriff_dataset = pd.read_csv("/data/us_imports_from_china_2019.csv")
tarriff_dataset.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'us_mfn_tariffs_india_2019.csv'

In [29]:
def load_and_preprocess_data():
    # Load the datasets
    china_tariffs = pd.read_csv('/data/us_mfn_tariffs_china_2019.csv')
    india_tariffs = pd.read_csv('/data/us_mfn_tariffs_india_2019.csv')
    china_imports = pd.read_csv('/data/us_imports_from_china_2019.csv')
    india_imports = pd.read_csv('/data/us_imports_from_india_2019.csv')

In [25]:
def preprocess_data(df):
    # Drop rows with missing values in key columns
    df = df.dropna(subset=['ProductOrSector', 'Value'])
        
    # Filter for the year 2019
    df = df[df['Year'] == 2019]
        
    # Ensure numeric values
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
        
    return df


# import pandas as pd  

# # Load data  
# df_india = pd.read_csv('/Users/amriteshanand/northeastern/INFO 6105 Data Science/Final Project/TariffShift/data/us_mfn_tariffs_india_2019.csv')  
# df_china = pd.read_csv('/Users/amriteshanand/northeastern/INFO 6105 Data Science/Final Project/TariffShift/data/us_imports_from_china_2019.csv')  

# # Clean numeric columns  
# df_india['Value'] = pd.to_numeric(df_india['Value'], errors='coerce')  
# df_china['Value'] = pd.to_numeric(df_china['Value'], errors='coerce')  

# # Standardize HS codes  
# df_india['HS Code'] = df_india['ProductOrSectorCode'].astype(str).str.strip()  
# df_china['HS Code'] = df_china['ProductOrSectorCode'].astype(str).str.strip()  

# df_india.head()
# df_china.head()

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
import streamlit as st
import os

# Set random seed for reproducibility
np.random.seed(42)

# Function to load and preprocess data
def load_and_preprocess_data():
    # Load the datasets
    china_tariffs = pd.read_csv('/data/us_mfn_tariffs_china_2019.csv')
    india_tariffs = pd.read_csv('/data/us_mfn_tariffs_india_2019.csv')
    china_imports = pd.read_csv('/data/us_imports_from_china_2019.csv')
    india_imports = pd.read_csv('/data/us_imports_from_india_2019.csv')
    
    # Clean and filter the datasets
    def preprocess_data(df):
        # Drop rows with missing values in key columns
        df = df.dropna(subset=['ProductOrSector', 'Value'])
        
        # Filter for the year 2019
        df = df[df['Year'] == 2019]
        
        # Ensure numeric values
        df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
        
        return df
    
    # Preprocess all datasets
    china_tariffs = preprocess_data(china_tariffs)
    india_tariffs = preprocess_data(india_tariffs)
    china_imports = preprocess_data(china_imports)
    india_imports = preprocess_data(india_imports)
    
    # Create aggregated views by product sector
    def aggregate_by_sector(df, value_col='Value'):
        return df.groupby('ProductOrSector')[value_col].mean().reset_index()
    
    china_tariffs_agg = aggregate_by_sector(china_tariffs)
    india_tariffs_agg = aggregate_by_sector(india_tariffs)
    china_imports_agg = aggregate_by_sector(china_imports, 'Value')
    india_imports_agg = aggregate_by_sector(india_imports, 'Value')
    
    # Merge datasets to create a combined view
    # First, merge tariff data
    tariff_comparison = pd.merge(
        china_tariffs_agg, 
        india_tariffs_agg, 
        on='ProductOrSector', 
        suffixes=('_china', '_india')
    )
    
    # Calculate tariff differential
    tariff_comparison['tariff_differential'] = tariff_comparison['Value_china'] - tariff_comparison['Value_india']
    
    # Merge with import data
    merged_data = pd.merge(
        tariff_comparison,
        china_imports_agg,
        on='ProductOrSector',
        suffixes=('', '_china_imports')
    )
    
    merged_data = pd.merge(
        merged_data,
        india_imports_agg,
        on='ProductOrSector',
        how='left',
        suffixes=('', '_india_imports')
    )
    
    # Rename columns for clarity
    merged_data = merged_data.rename(columns={
        'Value': 'china_imports_value',
        'Value_india_imports': 'india_imports_value'
    })
    
    # Fill NaN values for India imports with 0 (no current imports)
    merged_data['india_imports_value'] = merged_data['india_imports_value'].fillna(0)
    
    # Calculate import ratio and market share
    merged_data['total_imports'] = merged_data['china_imports_value'] + merged_data['india_imports_value']
    merged_data['china_market_share'] = merged_data['china_imports_value'] / merged_data['total_imports']
    merged_data['india_market_share'] = merged_data['india_imports_value'] / merged_data['total_imports']
    
    # Create potential shift score (combining tariff differential and import volume)
    merged_data['shift_potential_score'] = merged_data['tariff_differential'] * merged_data['china_imports_value'] / 1e6
    
    # Filter for sectors with positive tariff differential (China > India)
    potential_shift_sectors = merged_data[merged_data['tariff_differential'] > 0].sort_values(
        by='shift_potential_score', ascending=False
    )
    
    return china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data

# Function for exploratory data analysis and visualization
def perform_eda(potential_shift_sectors, output_dir='figures'):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Visualize tariff differentials
    plt.figure(figsize=(14, 8))
    sns.barplot(x='ProductOrSector', y='tariff_differential', 
                data=potential_shift_sectors.head(15))
    plt.xticks(rotation=90)
    plt.title('Top 15 Sectors by Tariff Differential (China vs. India)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/tariff_differential.png')
    plt.close()
    
    # Visualize import volumes
    plt.figure(figsize=(14, 8))
    plt.bar(potential_shift_sectors['ProductOrSector'].head(15), 
            potential_shift_sectors['china_imports_value'].head(15) / 1e6)
    plt.xticks(rotation=90)
    plt.title('US Import Volume from China (Top 15 Sectors with Tariff Differential, Million USD)')
    plt.ylabel('Million USD')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/china_import_volume.png')
    plt.close()
    
    # Visualize potential shift score
    plt.figure(figsize=(14, 8))
    plt.bar(potential_shift_sectors['ProductOrSector'].head(15), 
            potential_shift_sectors['shift_potential_score'].head(15))
    plt.xticks(rotation=90)
    plt.title('Manufacturing Shift Potential Score (Top 15 Sectors)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/shift_potential_score.png')
    plt.close()
    
    # Create scatter plot of tariff differential vs. import volume
    plt.figure(figsize=(12, 8))
    plt.scatter(potential_shift_sectors['tariff_differential'], 
                potential_shift_sectors['china_imports_value'] / 1e6,
                alpha=0.7)
    
    # Label the points for top 10 sectors
    for i, row in potential_shift_sectors.head(10).iterrows():
        plt.annotate(row['ProductOrSector'], 
                     xy=(row['tariff_differential'], row['china_imports_value'] / 1e6),
                     xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Tariff Differential (China - India, %)')
    plt.ylabel('US Import Volume from China (Million USD)')
    plt.title('Tariff Differential vs. Import Volume by Sector')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/tariff_vs_imports.png')
    plt.close()
    
    # Show market share comparison (China vs India)
    plt.figure(figsize=(14, 8))
    sectors = potential_shift_sectors['ProductOrSector'].head(10)
    china_share = potential_shift_sectors['china_market_share'].head(10)
    india_share = potential_shift_sectors['india_market_share'].head(10)
    
    x = np.arange(len(sectors))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(14, 8))
    rects1 = ax.bar(x - width/2, china_share, width, label='China')
    rects2 = ax.bar(x + width/2, india_share, width, label='India')
    
    ax.set_ylabel('Market Share')
    ax.set_title('US Import Market Share: China vs India (Top 10 Sectors)')
    ax.set_xticks(x)
    ax.set_xticklabels(sectors, rotation=90)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/market_share_comparison.png')
    plt.close()
    
    print(f"EDA visualizations saved to {output_dir}/")
    return True

# Function to build the PyMC predictive model
def build_pymc_model(potential_shift_sectors):
    # Prepare data for modeling
    model_data = potential_shift_sectors.copy()
    
    # Normalize the features for modeling
    model_data['tariff_diff_norm'] = (model_data['tariff_differential'] - 
                                      model_data['tariff_differential'].mean()) / model_data['tariff_differential'].std()
    model_data['china_imports_norm'] = (model_data['china_imports_value'] - 
                                       model_data['china_imports_value'].mean()) / model_data['china_imports_value'].std()
    model_data['india_imports_norm'] = (model_data['india_imports_value'] - 
                                       model_data['india_imports_value'].mean()) / model_data['india_imports_value'].std()
    
    # Create a PyMC model to predict shift probability
    with pm.Model() as shift_model:
        # Priors for unknown model parameters
        alpha = pm.Normal('alpha', mu=0, sigma=10)
        beta_tariff = pm.Normal('beta_tariff', mu=0, sigma=10)
        beta_china_imports = pm.Normal('beta_china_imports', mu=0, sigma=10)
        beta_india_imports = pm.Normal('beta_india_imports', mu=0, sigma=10)
        
        # Expected value of outcome
        mu = alpha + beta_tariff * model_data['tariff_diff_norm'] + \
             beta_china_imports * model_data['china_imports_norm'] + \
             beta_india_imports * model_data['india_imports_norm']
        
        # Likelihood (sampling distribution) of observations
        shift_probability = pm.Deterministic('shift_probability', pm.math.sigmoid(mu))
        
        # Add a Beta distribution to model the actual shift likelihood
        pm.Beta('observed_shift', alpha=1 + shift_probability * 10, 
                beta=1 + (1 - shift_probability) * 10, observed=None)
        
        # Sample from the posterior
        trace = pm.sample(2000, tune=1000, return_inferencedata=True, target_accept=0.9, compiler_kwargs={'cxx':''})
        
    # Extract the results
    with shift_model:
        posterior_samples = pm.sample_posterior_predictive(trace)
        
    # Get the mean shift probability for each sector
    sector_shift_probs = []
    for i, row in model_data.iterrows():
        sector_data = {
            'tariff_diff_norm': row['tariff_diff_norm'],
            'china_imports_norm': row['china_imports_norm'],
            'india_imports_norm': row['india_imports_norm']
        }
        
        with shift_model:
            mu = trace.posterior['alpha'].mean() + \
                 trace.posterior['beta_tariff'].mean() * sector_data['tariff_diff_norm'] + \
                 trace.posterior['beta_china_imports'].mean() * sector_data['china_imports_norm'] + \
                 trace.posterior['beta_india_imports'].mean() * sector_data['india_imports_norm']
            
            shift_prob = 1 / (1 + np.exp(-mu))
            
        sector_shift_probs.append({
            'ProductOrSector': row['ProductOrSector'],
            'shift_probability': shift_prob.item(),
            'tariff_differential': row['tariff_differential'],
            'china_imports_value': row['china_imports_value'],
            'india_imports_value': row['india_imports_value']
        })
    
    # Convert to DataFrame and sort by shift probability
    shift_predictions = pd.DataFrame(sector_shift_probs).sort_values(
        by='shift_probability', ascending=False
    )
    
    print("Top 10 sectors most likely to shift from China to India:")
    print(shift_predictions[['ProductOrSector', 'shift_probability', 
                            'tariff_differential', 'china_imports_value']].head(10))
    
    return shift_model, trace, shift_predictions

# Function to analyze model results and create visualizations
def analyze_model_results(shift_model, trace, shift_predictions, output_dir='figures'):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Plot posterior distributions for model parameters
    az.plot_posterior(trace, var_names=['alpha', 'beta_tariff', 'beta_china_imports', 'beta_india_imports'])
    plt.tight_layout()
    plt.savefig(f'{output_dir}/parameter_posteriors.png')
    plt.close()
    
    # Plot shift probability vs. tariff differential
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(shift_predictions['tariff_differential'], 
                shift_predictions['shift_probability'],
                s=shift_predictions['china_imports_value'] / 1e6,
                alpha=0.7)
    
    # Add labels for top sectors
    for i, row in shift_predictions.head(5).iterrows():
        plt.annotate(row['ProductOrSector'], 
                   xy=(row['tariff_differential'], row['shift_probability']),
                   xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Tariff Differential (China - India, %)')
    plt.ylabel('Predicted Shift Probability')
    plt.title('Manufacturing Shift Probability by Sector')
    
    # Add legend for bubble size
    handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, 
                                             num=4, func=lambda x: x * 1e6)
    legend = plt.legend(handles, labels, loc="upper right", title="Import Volume (USD)")
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/shift_probability_plot.png')
    plt.close()
    
    # Plot potential savings for top sectors
    top_sectors = shift_predictions.head(10)
    top_sectors['potential_savings'] = top_sectors['china_imports_value'] * top_sectors['tariff_differential'] / 100 / 1e6
    
    plt.figure(figsize=(14, 8))
    plt.bar(top_sectors['ProductOrSector'], top_sectors['potential_savings'])
    plt.xticks(rotation=90)
    plt.title('Potential Tariff Savings if Manufacturing Shifts (Million USD)')
    plt.ylabel('Million USD')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/potential_savings.png')
    plt.close()
    
    # Calculate and report key metrics for the top 5 sectors
    top5_sectors = shift_predictions.head(5)
    top5_sectors['market_size'] = top5_sectors['china_imports_value'] / 1e6  # In millions
    top5_sectors['india_current_share'] = top5_sectors['india_imports_value'] / (top5_sectors['china_imports_value'] + top5_sectors['india_imports_value'])
    top5_sectors['potential_savings'] = top5_sectors['china_imports_value'] * top5_sectors['tariff_differential'] / 100 / 1e6  # In millions
    
    # Create a sector priority matrix based on multiple factors
    sector_matrix = top5_sectors[['ProductOrSector', 'shift_probability', 'market_size', 
                                 'india_current_share', 'potential_savings']]
    
    # Save to CSV for reference
    sector_matrix.to_csv(f'{output_dir}/sector_priority_matrix.csv', index=False)
    
    print(f"Model analysis visualizations and reports saved to {output_dir}/")
    return sector_matrix

# Streamlit app for data visualization and interaction
def create_streamlit_app(potential_shift_sectors, shift_predictions):
    st.title('Manufacturing Shift Analysis: China to India')
    
    st.header('Tariff and Trade Analysis')
    
    # Add a summary of the analysis
    st.markdown("""
    This analysis explores the potential for manufacturing shifts from China to India based on tariff differentials 
    and trade data. We examine which sectors India should focus on developing to capitalize on the significant 
    tariff differentials between Chinese imports (145% in some sectors) and Indian imports (0% in many cases) 
    to the United States.
    """)
    
    # Display key metrics in a dashboard style
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric(
            label="Average Tariff Differential", 
            value=f"{potential_shift_sectors['tariff_differential'].mean():.2f}%"
        )
    
    with col2:
        st.metric(
            label="Total Import Volume from China", 
            value=f"${potential_shift_sectors['china_imports_value'].sum() / 1e9:.2f}B"
        )
    
    with col3:
        st.metric(
            label="Total Import Volume from India", 
            value=f"${potential_shift_sectors['india_imports_value'].sum() / 1e9:.2f}B"
        )
    
    st.subheader('Top Sectors by Tariff Differential')
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.barplot(x='ProductOrSector', y='tariff_differential', 
                data=potential_shift_sectors.head(15), ax=ax)
    plt.xticks(rotation=90)
    st.pyplot(fig)
    
    st.subheader('US Import Volume from China by Sector')
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.bar(potential_shift_sectors['ProductOrSector'].head(15), 
            potential_shift_sectors['china_imports_value'].head(15) / 1e6)
    plt.xticks(rotation=90)
    plt.ylabel('Million USD')
    st.pyplot(fig)
    
    st.header('Manufacturing Shift Predictions')
    
    st.subheader('Top 10 Sectors Most Likely to Shift from China to India')
    # Format the table for better readability
    display_df = shift_predictions[['ProductOrSector', 'shift_probability', 
                               'tariff_differential', 'china_imports_value']].head(10).copy()
    display_df['shift_probability'] = display_df['shift_probability'].apply(lambda x: f"{x:.2%}")
    display_df['tariff_differential'] = display_df['tariff_differential'].apply(lambda x: f"{x:.2f}%")
    display_df['china_imports_value'] = display_df['china_imports_value'].apply(lambda x: f"${x/1e6:.2f}M")
    display_df.columns = ['Product/Sector', 'Shift Probability', 'Tariff Differential', 'US Imports from China']
    
    st.table(display_df)
    
    st.subheader('Shift Probability vs. Tariff Differential')
    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.scatter(shift_predictions['tariff_differential'], 
                shift_predictions['shift_probability'],
                s=shift_predictions['china_imports_value'] / 1e6,
                alpha=0.7)
    
    # Add labels for top sectors
    for i, row in shift_predictions.head(5).iterrows():
        ax.annotate(row['ProductOrSector'], 
                   xy=(row['tariff_differential'], row['shift_probability']),
                   xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Tariff Differential (China - India, %)')
    plt.ylabel('Predicted Shift Probability')
    plt.title('Manufacturing Shift Probability by Sector')
    
    # Add legend for bubble size
    handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, 
                                             num=4, func=lambda x: x * 1e6)
    legend = ax.legend(handles, labels, loc="upper right", title="Import Volume (USD)")
    
    st.pyplot(fig)
    
    st.header('Priority Sectors for India')
    
    # Display the top 5 sectors with detailed information
    st.markdown("### Top 5 Priority Sectors for India to Focus On")
    
    for i, row in shift_predictions.head(5).iterrows():
        with st.expander(f"{i+1}. {row['ProductOrSector']}"):
            col1, col2 = st.columns(2)
            
            with col1:
                st.markdown(f"**Shift Probability:** {row['shift_probability']:.2%}")
                st.markdown(f"**Tariff Differential:** {row['tariff_differential']:.2f}%")
                st.markdown(f"**US Imports from China:** ${row['china_imports_value'] / 1e6:.2f} million")
            
            with col2:
                st.markdown(f"**US Imports from India:** ${row['india_imports_value'] / 1e6:.2f} million")
                current_share = row['india_imports_value'] / (row['china_imports_value'] + row['india_imports_value']) if (row['china_imports_value'] + row['india_imports_value']) > 0 else 0
                st.markdown(f"**Current India Market Share:** {current_share:.2%}")
                potential_savings = row['china_imports_value'] * row['tariff_differential'] / 100
                st.markdown(f"**Potential Tariff Savings if Shifted:** ${potential_savings / 1e6:.2f} million")
            
            # Add recommendations for this sector
            st.markdown("#### Development Recommendations:")
            st.markdown("""
            1. Establish specialized manufacturing zones
            2. Develop skilled workforce training programs
            3. Seek technology transfer partnerships
            4. Develop local supply chains
            5. Implement quality control systems aligned with US requirements
            """)
    
    st.header('Interactive Sector Explorer')
    
    selected_sector = st.selectbox('Select a sector to explore:', 
                                  options=shift_predictions['ProductOrSector'].tolist())
    
    if selected_sector:
        sector_data = shift_predictions[shift_predictions['ProductOrSector'] == selected_sector].iloc[0]
        
        st.write(f"### {selected_sector}")
        
        col1, col2 = st.columns(2)
        
        with col1:
            st.write(f"**Shift Probability:** {sector_data['shift_probability']:.2%}")
            st.write(f"**Tariff Differential:** {sector_data['tariff_differential']:.2f}%")
        
        with col2:
            st.write(f"**US Imports from China:** ${sector_data['china_imports_value'] / 1e6:.2f} million")
            st.write(f"**US Imports from India:** ${sector_data['india_imports_value'] / 1e6:.2f} million")
        
        # Calculate potential tariff savings
        potential_savings = sector_data['china_imports_value'] * sector_data['tariff_differential'] / 100
        st.write(f"**Potential Tariff Savings if Shifted:** ${potential_savings / 1e6:.2f} million")
        
        # Calculate and display additional metrics
        current_share = sector_data['india_imports_value'] / (sector_data['china_imports_value'] + sector_data['india_imports_value']) if (sector_data['china_imports_value'] + sector_data['india_imports_value']) > 0 else 0
        st.write(f"**Current India Market Share:** {current_share:.2%}")
        
        # Show potential growth opportunity
        growth_potential = sector_data['china_imports_value'] * sector_data['shift_probability']
        st.write(f"**Potential Growth Opportunity:** ${growth_potential / 1e6:.2f} million")
    
    # Conclusion section
    st.header('Conclusion and Recommendations')
    
    st.markdown("""
    ### Key Sectors India Should Focus On

    1. **Electronics and Electronic Components**: Highest shift potential score with substantial tariff differentials and high import volumes.
    
    2. **Textiles and Apparel**: Leverage existing manufacturing capabilities and significant tariff advantages.
    
    3. **Machinery and Mechanical Appliances**: Strong potential with moderate to high tariff differentials.
    
    4. **Pharmaceuticals and Medical Equipment**: Build on existing pharmaceutical industry strengths.
    
    5. **Furniture and Home Goods**: Capitalize on labor-intensive manufacturing opportunities with favorable tariff conditions.

    ### Strategic Recommendations

    1. **Create Specialized Manufacturing Zones** with tailored infrastructure for priority sectors
    
    2. **Develop Skilled Workforce** through targeted training programs
    
    3. **Facilitate Technology Transfer** through strategic partnerships and joint ventures
    
    4. **Build Robust Supply Chains** within India to maximize value addition
    
    5. **Implement Quality Standards** aligned with US market requirements
    """)
    
    # Add a footer
    st.markdown("---")
    st.markdown("© 2025 Manufacturing Shift Analysis | Data Sources: US MFN Tariffs and Trade Data 2019")

# Main function to run the complete analysis
def main():
    print("Starting Manufacturing Shift Analysis...")
    
    # Step 1: Load and preprocess data
    print("Loading and preprocessing data...")
    china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data = load_and_preprocess_data()
    
    # Step 2: Perform exploratory data analysis
    print("Performing exploratory data analysis...")
    perform_eda(potential_shift_sectors)
    
    # Step 3: Build PyMC model
    print("Building PyMC model...")
    shift_model, trace, shift_predictions = build_pymc_model(potential_shift_sectors)
    
    # Step 4: Analyze model results
    print("Analyzing model results...")
    sector_matrix = analyze_model_results(shift_model, trace, shift_predictions)
    
    # Step 5: Create Streamlit app for visualization
    print("To run the Streamlit app, execute:")
    print("streamlit run app.py")
    
    # Export prediction results to CSV
    shift_predictions.to_csv('manufacturing_shift_predictions.csv', index=False)
    print("Analysis complete! Prediction results saved to manufacturing_shift_predictions.csv")
    
    # Print the top 5 priority sectors for India with detailed metrics
    print("\nTOP 5 PRIORITY SECTORS FOR INDIA TO FOCUS ON:")
    print("=============================================")
    
    for i, row in sector_matrix.iterrows():
        print(f"{i+1}. {row['ProductOrSector']}")
        print(f"   Shift Probability: {row['shift_probability']:.2%}")
        print(f"   Market Size: ${row['market_size']:.2f}M")
        print(f"   Current India Market Share: {row['india_current_share']:.2%}")
        print(f"   Potential Tariff Savings: ${row['potential_savings']:.2f}M")
        print("---------------------------------------------")

# App entry point for Streamlit
def streamlit_entry():
    # Load and preprocess data
    china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data = load_and_preprocess_data()
    
    # Load predictions or generate them if not available
    try:
        shift_predictions = pd.read_csv('manufacturing_shift_predictions.csv')
        print("Loaded existing predictions from file")
    except:
        print("Generating new predictions...")
        _, _, shift_predictions = build_pymc_model(potential_shift_sectors)
    
    # Create and display the Streamlit app
    create_streamlit_app(potential_shift_sectors, shift_predictions)

if __name__ == "__main__":
    # If run directly, execute the full analysis
    main()
    
    # To run the Streamlit app only, comment out the above line and uncomment below:
    # streamlit_entry()

Starting Manufacturing Shift Analysis...
Loading and preprocessing data...
Performing exploratory data analysis...
EDA visualizations saved to figures/
Building PyMC model...


ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(1.0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/re

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(1.0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/re

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(1.0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/re

Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(1.0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/l

...

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Cast{float32}(10)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewritin

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Cast{float64}(10)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewritin

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ScalarFromTensor(0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewrit

ImportError: dlopen(/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpraytmi2s/m592274835bee3bcf7bf2cb65b70477fb2a0214df94b73fac749e4ac5fb9f3e01.so, 0x0002): tried: '/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpraytmi2s/m592274835bee3bcf7bf2cb65b70477fb2a0214df94b73fac749e4ac5fb9f3e01.so' (mach-o file, but is an incompatible architecture (have 'x86_64', need 'arm64e' or 'arm64')), '/System/Volumes/Preboot/Cryptexes/OS/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpraytmi2s/m592274835bee3bcf7bf2cb65b70477fb2a0214df94b73fac749e4ac5fb9f3e01.so' (no such file), '/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpraytmi2s/m592274835bee3bcf7bf2cb65b70477fb2a0214df94b73fac749e4ac5fb9f3e01.so' (mach-o file, but is an incompatible architecture (have 'x86_64', need 'arm64e' or 'arm64'))
Apply node that ca

<Figure size 1400x800 with 0 Axes>

In [33]:
plt.figure(figsize=(10, 6))  
plt.scatter(  
    df_merged['Tariff_Diff'], df_merged['Value_china'],  
    alpha=0.6, color='tab:blue'  
)  
plt.xlabel('Tariff Differential (%)')  
plt.ylabel('US Imports from China (USD)')  
plt.title('Trade Value vs. Tariff Differential by Sector')  
plt.grid(True)  
plt.show() 

NameError: name 'df_merged' is not defined

<Figure size 1000x600 with 0 Axes>

In [35]:
# import pymc as pm  

with pm.Model() as logistic_model:  
    # Priors  
    alpha = pm.Normal('alpha', mu=0, sigma=10)  
    beta_trade = pm.Normal('beta_trade', mu=0, sigma=10)  

    # Logistic link  
    p = pm.math.sigmoid(  
        alpha + beta_trade * df_merged['Log_Value_China']  
    )  
    # Likelihood  
    obs = pm.Bernoulli(  
        'obs', p=p, observed=df_merged['Shift_Candidate']  
    )  
    # MAP estimation  
    map_estimate = pm.find_MAP()

NameError: name 'df_merged' is not defined

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(-inf)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/r

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(1)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewr

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Cast{float32}(10)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewritin

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Cast{float64}(10)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewritin

...

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Cast{float64}(10)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewritin

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Gt(10, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(-1.0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/r

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: ExpandDims{axis=0}(0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1922, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewriting/basic.py", line 1086, in transform
    return self.fn(fgraph, node)
           ~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/tensor/rewriting/basic.py", line 1173, in constant_folding
    return unconditional_constant_folding.transform(fgraph, node)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pymc_ds/lib/python3.13/site-packages/pytensor/graph/rewr

ImportError: dlopen(/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpz3jq710d/m205e7b16bc57388bf8d2aae5aba0e1b7d306f1edf0edcd8da345a42af4834d89.so, 0x0002): tried: '/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpz3jq710d/m205e7b16bc57388bf8d2aae5aba0e1b7d306f1edf0edcd8da345a42af4834d89.so' (mach-o file, but is an incompatible architecture (have 'x86_64', need 'arm64e' or 'arm64')), '/System/Volumes/Preboot/Cryptexes/OS/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpz3jq710d/m205e7b16bc57388bf8d2aae5aba0e1b7d306f1edf0edcd8da345a42af4834d89.so' (no such file), '/Users/amriteshanand/.pytensor/compiledir_macOS-15.3.2-arm64-i386-64bit-Mach-O-i386-3.13.2-64/tmpz3jq710d/m205e7b16bc57388bf8d2aae5aba0e1b7d306f1edf0edcd8da345a42af4834d89.so' (mach-o file, but is an incompatible architecture (have 'x86_64', need 'arm64e' or 'arm64'))
Apply node that ca

In [37]:
!pip install streamlit
import streamlit as st  

st.title("Manufacturing Shift Analysis")  
selected_sector = st.sidebar.selectbox(  
    'Select Sector', df_merged['ProductOrSector']  
)  
sector_data = df_merged[  
    df_merged['ProductOrSector'] == selected_sector  
]  

# Plot trade value vs. tariff differential  
fig, ax = plt.subplots()  
ax.scatter(  
    sector_data['Tariff_Diff'], sector_data['Value_china'],  
    color='tab:blue'  
)  
st.pyplot(fig)  

# Display model coefficients  
st.write("MAP Estimates:", map_estimate) 


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


2025-04-15 15:34:38.781 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-15 15:34:38.814 
  command:

    streamlit run /opt/anaconda3/envs/pymc5env/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-04-15 15:34:38.814 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name 'df_merged' is not defined

In [39]:
import pymc as pm

with pm.Model() as probit_model:
    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta_trade = pm.Normal("beta_trade", mu=0, sigma=10)
    
    # Probit link function
    mu = alpha + beta_trade * df_merged["Log_Value_China"]
    p_shift = pm.math.invprobit(mu)  # Inverse probit (Φ)
    
    # Likelihood
    obs = pm.Bernoulli("obs", p=p_shift, observed=df_merged["Shift_Candidate"])
    
    # MAP estimation
    map_estimate = pm.find_MAP()

NameError: name 'df_merged' is not defined